# Custom Training Data and Inference
This notebook demonstrates how to use your own training data to train the MLE model and then perform inference on test data.

In [ ]:
from src.MLE import MLE
from src.estimation import estimate_text_distribution
import os
import pandas as pd

## Step 1: Prepare Your Training Data

Your training data should be organized as follows:
- Two separate parquet files: one for human-generated text and one for AI-generated text
- Each file should contain a column with tokenized sentences

### Data Format Example:

**human_training_data.parquet:**
```
| sentence |
|----------|
| ["This", "is", "human", "text"] |
| ["Another", "human", "example"] |
```

**ai_training_data.parquet:**
```
| sentence |
|----------|
| ["This", "is", "AI", "generated"] |
| ["Another", "AI", "example"] |
```

## Step 2: Configure Paths to Your Training Data

In [ ]:
# ===== MODIFY THESE PATHS TO YOUR OWN TRAINING DATA =====
# Path to human-generated training data (parquet file with tokenized sentences)
HUMAN_TRAINING_PATH = "data/your_training_data/human_training_data.parquet"

# Path to AI-generated training data (parquet file with tokenized sentences)
AI_TRAINING_PATH = "data/your_training_data/ai_training_data.parquet"

# Path where the learned distribution model will be saved
MODEL_OUTPUT_PATH = "distribution/your_custom_model.parquet"

# Path to test data (parquet file with 'inference_sentence' column)
TEST_DATA_PATH = "data/your_test_data/test_data.parquet"

# ===== OPTIONAL: Verification =====
print(f"Human training data path: {HUMAN_TRAINING_PATH}")
print(f"AI training data path: {AI_TRAINING_PATH}")
print(f"Model output path: {MODEL_OUTPUT_PATH}")
print(f"Test data path: {TEST_DATA_PATH}")

## Step 3: Train the Model (Estimate Text Distribution)

In [ ]:
# Check if training data files exist
if os.path.exists(HUMAN_TRAINING_PATH) and os.path.exists(AI_TRAINING_PATH):
    print("Training data files found. Starting distribution estimation...")
    print(f"Human training data: {HUMAN_TRAINING_PATH}")
    print(f"AI training data: {AI_TRAINING_PATH}")
    
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(MODEL_OUTPUT_PATH), exist_ok=True)
    
    # Estimate the text distribution from your training data
    # This function computes word occurrence frequencies from human and AI-generated text
    estimate_text_distribution(
        human_parquet_path=HUMAN_TRAINING_PATH,
        ai_parquet_path=AI_TRAINING_PATH,
        output_parquet_path=MODEL_OUTPUT_PATH
    )
    
    print(f"\n✓ Distribution estimation completed!")
    print(f"Model saved to: {MODEL_OUTPUT_PATH}")
else:
    print("❌ Training data files not found!")
    print(f"Please ensure the following files exist:")
    print(f"  1. {HUMAN_TRAINING_PATH}")
    print(f"  2. {AI_TRAINING_PATH}")

## Step 4: Perform Inference on Test Data

In [ ]:
# Check if model file and test data exist
if os.path.exists(MODEL_OUTPUT_PATH):
    # Load the trained model
    model = MLE(MODEL_OUTPUT_PATH)
    print(f"✓ Model loaded from: {MODEL_OUTPUT_PATH}")
    
    if os.path.exists(TEST_DATA_PATH):
        print(f"\nPerforming inference on test data: {TEST_DATA_PATH}")
        print("This may take a few minutes depending on the size of your test data...\n")
        
        # Perform inference
        # Returns:
        #   alpha: estimated proportion of AI-modified/generated text (0 to 1)
        #   ci: confidence interval for the estimate
        alpha, ci = model.inference(TEST_DATA_PATH)
        
        # Display results
        print("="*50)
        print("INFERENCE RESULTS")
        print("="*50)
        print(f"Estimated Alpha (AI-modified text proportion): {alpha:.4f}")
        print(f"Confidence Interval: {ci:.4f}")
        print(f"Percentage: {alpha*100:.2f}% ± {ci*100:.2f}%")
        print("="*50)
    else:
        print(f"❌ Test data file not found: {TEST_DATA_PATH}")
else:
    print(f"❌ Model file not found: {MODEL_OUTPUT_PATH}")
    print("Please complete Step 3 (training) first.")

## Step 5: Batch Testing on Multiple Test Sets (Optional)

In [ ]:
# Example: If you have multiple test datasets, test them all

# Define multiple test datasets
test_datasets = {
    "test_set_1": "data/your_test_data/test_set_1.parquet",
    "test_set_2": "data/your_test_data/test_set_2.parquet",
    "test_set_3": "data/your_test_data/test_set_3.parquet",
}

if os.path.exists(MODEL_OUTPUT_PATH):
    model = MLE(MODEL_OUTPUT_PATH)
    results = {}
    
    for dataset_name, dataset_path in test_datasets.items():
        if os.path.exists(dataset_path):
            print(f"\nTesting: {dataset_name}...")
            alpha, ci = model.inference(dataset_path)
            results[dataset_name] = {"alpha": alpha, "ci": ci}
            print(f"  Alpha: {alpha:.4f} ± {ci:.4f}")
        else:
            print(f"⚠ Test dataset not found: {dataset_path}")
    
    # Display summary
    if results:
        print("\n" + "="*50)
        print("SUMMARY OF ALL TESTS")
        print("="*50)
        for dataset_name, metrics in results.items():
            print(f"{dataset_name:20s}: {metrics['alpha']:.4f} ± {metrics['ci']:.4f}")
        print("="*50)

## Step 6: Data Validation (Optional)

If you have test data with known ground truth alpha values, validate the model's accuracy:

In [ ]:
# Validation example: test the model on data with known ground truth alpha values

validation_alphas = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25]
validation_base_path = "data/your_validation_data/ground_truth_alpha_"

if os.path.exists(MODEL_OUTPUT_PATH):
    model = MLE(MODEL_OUTPUT_PATH)
    
    print("\nVALIDATION RESULTS")
    print("="*70)
    print(f"{'Ground Truth':>12} | {'Prediction':>12} | {'CI':>10} | {'Error':>10}")
    print("="*70)
    
    for alpha in validation_alphas:
        validation_path = f"{validation_base_path}{alpha}.parquet"
        
        if os.path.exists(validation_path):
            estimated, ci = model.inference(validation_path, exploded_data=True)
            error = abs(estimated - alpha)
            print(f"{alpha:12.3f} | {estimated:12.4f} | {ci:10.4f} | {error:10.4f}")
        else:
            print(f"⚠ Validation file not found: {validation_path}")
    
    print("="*70)
else:
    print(f"❌ Model file not found: {MODEL_OUTPUT_PATH}")

## Notes and Tips

### Data Preparation:
- Both training and test data must be in **parquet format**
- Sentences should be **tokenized** before being used
- Use the `tokenize_demo.ipynb` notebook as reference for tokenization

### Training Data:
- The more training data you have, the better the model performs
- Ensure human and AI data are well-balanced
- Training time depends on the size of your data

### Inference:
- Alpha ranges from 0 to 1 (0% to 100% AI-modified content)
- Lower CI values indicate higher confidence in the estimate
- Each inference typically takes 5-10 seconds for standard datasets

### Column Names:
- Training data: column name should be `sentence` or `human_sentence`/`ai_sentence`
- Test data: column name must be `inference_sentence`

For more details, check the project README.md and the src/estimation.py file.